# Lab 09: Foundry Tool Catalog

When creating agents, you can attach tools directly in the agent definition. This **ties the tool configuration to the agent** — every agent that needs the tool must include the full configuration, including any credentials:

```python
# Direct approach: credentials and endpoint embedded in agent definition
agent = project_client.agents.create_version(
    agent_name="contoso-pmo-agent",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions="...",
        tools=[{
            "type": "mcp",
            "server_label": "contoso_pmo_kb",
            "server_url": "https://func-contoso-pmo-mcp-xxxxxx.azurewebsites.net/runtime/webhooks/mcp/sse?code=<key>",
            "require_approval": "never"
        }]
    )
)
```

The **Foundry Tool Catalog** solves this by providing a centralized registry. Register your MCP servers and credentials once, then reference them by connection ID:

```python
# Catalog approach: reference tool by connection ID
contoso_pmo_tool = project_client.connections.get("contoso-pmo-mcp")

agent = project_client.agents.create_version(
    agent_name="contoso-pmo-catalog-agent",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions="...",
        tools=[{
            "type": "mcp",
            "server_label": "contoso_pmo_kb",
            "server_url": contoso_pmo_tool.target,           # From catalog
            "project_connection_id": contoso_pmo_tool.id,    # Reference to catalog entry
            "require_approval": "never"
        }]
    )
)
```

## Why Use the Tool Catalog?

| Direct Embedding | Tool Catalog |
|------------------|---------------|
| Tool config hardcoded in each agent | Register once, reference by ID |
| Rotating credentials = update every agent | Rotate credentials in one place |
| No visibility into which tools exist | Centralized registry of approved tools |
| Teams duplicate tool configurations | Teams share approved tools |
| Credentials scattered across agent definitions | Credentials stored securely in connections |

## Lab Flow

This lab registers the **Contoso PMO KB MCP server** — the 37-tool Azure Functions app built in lab 09-01 — into the Foundry Tool Catalog, then creates a Contoso PMO KB agent that consumes it via the catalog entry.

1. **Setup** — derive the Contoso PMO function app name and retrieve its MCP key
2. **Register Tools** — add Contoso PMO MCP to the catalog
3. **List Tools** — view all registered tools
4. **Use in Agent** — create a Contoso PMO KB agent that references the catalog entry

## Prerequisites

1. **Python environment** — run `uv sync` from the repository root, then select the `.venv` kernel in VS Code.
2. **Setup lab** — run `09-01-mcp-agent-setup.ipynb` first to deploy the Azure Functions MCP server.
3. **`.env` file** — the `.env` lives at the repository root. Ensure it contains:
   - `CONTOSO_PMO_FOUNDRY_PROJECT_ENDPOINT` — your Foundry project endpoint URL
   - `CONTOSO_PMO_RESOURCE_GROUP` — resource group of the Foundry AI Services account
   - `CHAT_MODEL` *(optional, default `gpt-4.1-mini`)*
   - `CONTOSO_PMO_MCP_RESOURCE_GROUP` *(optional, default `rg-foundry-contoso-pmo-mcp`)*
4. **Azure CLI** — run `az login` before executing the cells.

---

## Setup

Load `.env` from the repository root and read the required environment variables.

In [ ]:
import hashlib
import json
import os
import subprocess
import requests
from pathlib import Path
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from dotenv import load_dotenv
from IPython.display import display, Markdown

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
load_dotenv(repo_root / '.env', override=True)

PROJECT_ENDPOINT = os.environ['CONTOSO_PMO_FOUNDRY_PROJECT_ENDPOINT']
CHAT_MODEL       = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

# Derive account and project names from the endpoint URL.
# e.g. https://<account>.services.ai.azure.com/api/projects/<project>
_host        = PROJECT_ENDPOINT.split('/api/projects/')[0].replace('https://', '')
ACCOUNT_NAME = _host.split('.')[0]
PROJECT_NAME = PROJECT_ENDPOINT.split('/api/projects/')[-1]

# Subscription and resource group (used to construct the ARM API base URL)
SUBSCRIPTION_ID = (
    os.environ.get('AZURE_SUBSCRIPTION_ID')
    or json.loads(subprocess.run(
        'az account show -o json', shell=True, capture_output=True, text=True
    ).stdout)['id']
)
RESOURCE_GROUP = os.environ['CONTOSO_PMO_RESOURCE_GROUP']

# ARM REST API — used to register and list Tool Catalog connections
credential = DefaultAzureCredential()
_token = credential.get_token('https://management.azure.com/.default')
ARM_BASE = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}'
    f'/providers/Microsoft.CognitiveServices/accounts/{ACCOUNT_NAME}/projects/{PROJECT_NAME}'
)
API_VERSION = '2025-04-01-preview'
headers = {'Authorization': f'Bearer {_token.token}', 'Content-Type': 'application/json'}

# Contoso PMO MCP server — same naming convention as 09-01-mcp-agent-setup.ipynb
CONTOSO_PMO_MCP_RG     = os.environ.get('CONTOSO_PMO_MCP_RESOURCE_GROUP', 'rg-foundry-contoso-pmo-mcp')
_suffix           = hashlib.md5(f'{SUBSCRIPTION_ID}-{CONTOSO_PMO_MCP_RG}'.encode()).hexdigest()[:6]
CONTOSO_PMO_FUNC_APP_NAME = os.environ.get('CONTOSO_PMO_FUNC_APP_NAME') or f'func-contoso-pmo-mcp-{_suffix}'

print(f'Project endpoint  : {PROJECT_ENDPOINT}')
print(f'Model             : {CHAT_MODEL}')
print(f'Contoso PMO func app   : {CONTOSO_PMO_FUNC_APP_NAME}')

In [ ]:
# Retrieve the mcp_extension system key from the deployed Contoso PMO function app.
# This key is generated by the Azure Functions MCP extension after first load.
_r = subprocess.run(
    f'az functionapp keys list -g "{CONTOSO_PMO_MCP_RG}" -n "{CONTOSO_PMO_FUNC_APP_NAME}" '
    f'--query "systemKeys.mcp_extension" -o tsv',
    shell=True, capture_output=True, text=True,
)
CONTOSO_PMO_MCP_KEY = _r.stdout.strip()

assert CONTOSO_PMO_MCP_KEY and CONTOSO_PMO_MCP_KEY != 'None', (
    f"MCP key not found for '{CONTOSO_PMO_FUNC_APP_NAME}' in '{CONTOSO_PMO_MCP_RG}'. "
    'Run 09-01-mcp-agent-setup.ipynb first.'
)

CONTOSO_PMO_MCP_SSE_URL = (
    f'https://{CONTOSO_PMO_FUNC_APP_NAME}.azurewebsites.net'
    f'/runtime/webhooks/mcp/sse?code={CONTOSO_PMO_MCP_KEY}'
)

print(f'Function app : {CONTOSO_PMO_FUNC_APP_NAME}')
print(f'MCP SSE URL  : https://{CONTOSO_PMO_FUNC_APP_NAME}.azurewebsites.net/runtime/webhooks/mcp/sse?code=<key>')

---

## Part 1: Register the Contoso PMO MCP Server

Register the Contoso PMO KB MCP server in the Tool Catalog using the ARM REST API. The registration stores the full SSE URL (including the Azure Functions system key) as the connection `target` — any agent in this project can then reference it by connection ID without handling the key directly.

In [ ]:
def register_mcp_tool(name: str, endpoint_url: str):
    """Register an MCP server in the Tool Catalog."""
    url = f'{ARM_BASE}/connections/{name}?api-version={API_VERSION}'
    payload = {
        'properties': {
            'category': 'RemoteTool',
            'authType': 'None',
            'target': endpoint_url,
            'metadata': {'type': 'custom_MCP'},
        }
    }
    response = requests.put(url, headers=headers, json=payload)
    return response.status_code in [200, 201]

print('Helper functions ready')

In [ ]:
# Register the Contoso PMO KB MCP server.
# The Azure Functions MCP extension uses a system key embedded in the SSE URL (?code=...).
# The full URL (key included) is stored as the catalog target with authType "None" —
# the same pattern used when calling MCPTool directly in the setup lab.
CONTOSO_PMO_MCP_TOOL_NAME = 'contoso-pmo-mcp'

if register_mcp_tool(CONTOSO_PMO_MCP_TOOL_NAME, CONTOSO_PMO_MCP_SSE_URL):
    print(f'Registered MCP tool: {CONTOSO_PMO_MCP_TOOL_NAME}')
    print(f'  Function app : {CONTOSO_PMO_FUNC_APP_NAME}')
    print(f'  Endpoint     : https://{CONTOSO_PMO_FUNC_APP_NAME}.azurewebsites.net/runtime/webhooks/mcp/sse?code=<key>')
    print(f'  Tools        : 37 (projects, people, tasks, meetings, risks, documents)')
else:
    print(f'Failed to register {CONTOSO_PMO_MCP_TOOL_NAME}')

---

## Part 2: List All Tools in the Catalog

View all registered MCP servers. Tools are identified by `category: RemoteTool` and `metadata.type: custom_MCP`.

In [ ]:
def list_mcp_tools():
    """List all MCP servers registered in the Foundry Tool Catalog."""
    url = f'{ARM_BASE}/connections?api-version={API_VERSION}'
    response = requests.get(url, headers=headers)
    connections = response.json().get('value', [])

    mcp_tools = [
        {'name': conn['name'], 'target': conn.get('properties', {}).get('target', '')}
        for conn in connections
        if conn.get('properties', {}).get('category') == 'RemoteTool'
        and conn.get('properties', {}).get('metadata', {}).get('type') == 'custom_MCP'
    ]
    return mcp_tools

mcp_tools = list_mcp_tools()
print(f'MCP Servers ({len(mcp_tools)}):')
for tool in mcp_tools:
    print(f'  - {tool["name"]}: {tool["target"]}')

In [ ]:
import uuid
import base64

sub_bytes   = uuid.UUID(SUBSCRIPTION_ID).bytes
encoded_sub = base64.urlsafe_b64encode(sub_bytes).decode('utf-8').rstrip('=')

TOOLS_PORTAL_URL = (
    f'https://ai.azure.com/nextgen/r/{encoded_sub},{RESOURCE_GROUP},,{ACCOUNT_NAME},{PROJECT_NAME}'
    f'/Build/tools'
)

display(Markdown(f"""
### View Tools in Foundry Portal

**[Open Tool Catalog in Portal]({TOOLS_PORTAL_URL})**

The portal provides a visual interface to browse and manage registered MCP servers and OpenAPI connections.
"""))

---

## Part 3: Use a Tool in an Agent

Create a Contoso PMO KB agent that uses the cataloged MCP tool. The key benefit: **no need to pass the MCP key** — the catalog connection stores the full SSE URL and the agent simply references it by connection ID.

In [ ]:
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# Retrieve the Contoso PMO MCP tool from the catalog
contoso_pmo_tool = project_client.connections.get(CONTOSO_PMO_MCP_TOOL_NAME)

print(f'Retrieved tool from catalog:')
print(f'  Name         : {contoso_pmo_tool.name}')
print(f'  Endpoint     : https://{CONTOSO_PMO_FUNC_APP_NAME}.azurewebsites.net/runtime/webhooks/mcp/sse?code=<key>')
print(f'  Connection ID: {contoso_pmo_tool.id}')

In [ ]:
CONTOSO_PMO_AGENT_NAME = 'contoso-pmo-catalog-agent'

_INSTRUCTIONS = (
    'You are the Contoso PMO knowledge base assistant. You help project managers '
    'and cross-functional team members in a consumer product launch environment.\n\n'
    'KNOWLEDGE BASE TOOLS\n'
    'You have 37 MCP tools covering projects, people, meetings, tasks, risks, '
    'documents, and distribution lists. Always use these tools to read and write '
    'data — never guess or fabricate IDs, names, or dates.\n\n'
    'GATE FRAMEWORK\n'
    'Projects follow the BLAST gate framework: G0 (concept) → G1 (feasibility) → '
    'G2 (development) → G3 (validation) → G4 (launch readiness) → G5 (launch) → '
    'G6 (post-launch review) → G7 (closure).\n\n'
    'LESSONS LEARNED\n'
    'When discussing project risks or planning for an upcoming gate, proactively '
    'search lessons learned (search_lessons, search_risk_patterns) to surface '
    'relevant past experience from completed projects.'
)

agent = project_client.agents.create_version(
    agent_name=CONTOSO_PMO_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_MODEL,
        instructions=_INSTRUCTIONS,
        tools=[
            {
                'type': 'mcp',
                'server_label': 'contoso_pmo_kb',
                'server_url': contoso_pmo_tool.target,           # From catalog
                'project_connection_id': contoso_pmo_tool.id,    # Reference to catalog entry
                'require_approval': 'never',
            }
        ],
    ),
    description='Contoso PMO KB assistant — MCP tool sourced from the Tool Catalog.',
)

print(f'Agent created: {agent.name} v{agent.version}')
print(f'  MCP tool from catalog: {contoso_pmo_tool.name}')

---

## Part 4: Query the agent

In [ ]:
openai_client = project_client.get_openai_client()


def ask(question: str) -> str:
    """Send a question to the cataloged Contoso PMO KB agent and return the response text."""
    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=question,
        extra_body={'agent_reference': {'name': agent.name, 'version': agent.version, 'type': 'agent_reference'}},
    )
    return response.output_text


print('OpenAI client ready. Run the query cells below.')

In [ ]:
print(ask(
    "What tasks are currently overdue across all projects? "
    "Who is responsible for each, and which project do they belong to, and what is the due date?"
))

In [ ]:
print(ask(
    "I'm planning the Aurora G4 gate review. "
    "Search for relevant lessons learned from past projects and identify any risk "
    "patterns around supplier or regulatory issues that I should be aware of."
))

In [ ]:
print(ask(
    "Search for documents containing discussion of scope changes or variant additions. "
    "Summarise the key decisions made and who was involved."
))

---

## Summary

The Contoso PMO MCP server is registered as a `RemoteTool` connection with `authType: "None"`. The Azure Functions system key is embedded in the SSE URL and stored as the connection `target` — agents reference the catalog entry by ID and never handle the key directly.

> **Key rotation**: retrieve the new `mcp_extension` key and call `register_mcp_tool("contoso-pmo-mcp", new_url)` again — the `PUT` is idempotent and all agents pick up the updated URL automatically.